In [105]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [136]:
from bs4 import BeautifulSoup
import pandas as pd

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 정보가 포함된 li 태그 찾기
    for paper in soup.find_all('li', class_='entry'):
        # 제목 찾기
        title_tag = paper.find('span', class_='title')
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        author_tags = paper.find_all('span', itemprop='author')
        authors_cleaned = ", ".join([author.text.strip() for author in author_tags]) if author_tags else "Unknown"

        # PDF 링크 만들기
        pdf_link = None
        
        # ACL Anthology 링크 찾기
        aclanthology_link = paper.find('a', href=True)
        if aclanthology_link and "aclanthology.org" in aclanthology_link['href']:
            pdf_link = aclanthology_link['href'] + ".pdf"
        
        # DOI 링크 찾기
        elif aclanthology_link and "doi.org" in aclanthology_link['href']:
            # DOI 링크에서 논문 코드만 추출
            doi_link = aclanthology_link['href']
            paper_code = doi_link.split('/')[-1]  # DOI에서 논문 코드만 추출
            pdf_link = f"https://aclanthology.org/{paper_code}.pdf"
        
        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [127]:
url = 'https://dblp.org/db/conf/acl/acl2019-1.html/'
DB_PATH = "con_db/ACL_conference_2019.db"
conference_name = 'ACL 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [108]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [137]:
df_papers = get_www_papers('html/ACL_2019_accepted_papers.html', conference_name)

In [138]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 57th Conference of the Asso...,"Anna Korhonen, David R. Traum, Lluís Màrquez",https://aclanthology.org/volumes/P19-1/.pdf,None,ACL 2019
1,One Time of Interaction May Not Be Enough: Go ...,"Chongyang Tao, Wei Wu, Can Xu, Wenpeng Hu, Don...",https://aclanthology.org/p19-1001.pdf,None,ACL 2019
2,Incremental Transformer with Deliberation Deco...,"Zekang Li, Cheng Niu, Fandong Meng, Yang Feng,...",https://aclanthology.org/p19-1002.pdf,None,ACL 2019
3,Improving Multi-turn Dialogue Modelling with U...,"Hui Su, Xiaoyu Shen, Rongzhi Zhang, Fei Sun, P...",https://aclanthology.org/p19-1003.pdf,None,ACL 2019
4,Do Neural Dialog Systems Use the Conversation ...,"Chinnadhurai Sankar, Sandeep Subramanian, Chri...",https://aclanthology.org/p19-1004.pdf,None,ACL 2019


In [139]:
df_papers = df_papers.drop(index=[0])

In [140]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,One Time of Interaction May Not Be Enough: Go ...,"Chongyang Tao, Wei Wu, Can Xu, Wenpeng Hu, Don...",https://aclanthology.org/p19-1001.pdf,None,ACL 2019
2,Incremental Transformer with Deliberation Deco...,"Zekang Li, Cheng Niu, Fandong Meng, Yang Feng,...",https://aclanthology.org/p19-1002.pdf,None,ACL 2019
3,Improving Multi-turn Dialogue Modelling with U...,"Hui Su, Xiaoyu Shen, Rongzhi Zhang, Fei Sun, P...",https://aclanthology.org/p19-1003.pdf,None,ACL 2019
4,Do Neural Dialog Systems Use the Conversation ...,"Chinnadhurai Sankar, Sandeep Subramanian, Chri...",https://aclanthology.org/p19-1004.pdf,None,ACL 2019
5,Boosting Dialog Response Generation.,"Wenchao Du, Alan W. Black",https://aclanthology.org/p19-1005.pdf,None,ACL 2019


In [141]:
save_to_database(df_papers, conference_name, DB_PATH) 

660개의 논문이 ACL 2019에 저장되었습니다.


# 2018

In [142]:
url = 'https://dblp.org/db/conf/acl/acl2018-1.html'
DB_PATH = "con_db/ACL_conference_2018.db"
conference_name = 'ACL 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [143]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [144]:
df_papers = get_www_papers('html/ACL_2018_accepted_papers.html', conference_name)

In [145]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 56th Annual Meeting of the ...,"Iryna Gurevych, Yusuke Miyao",https://aclanthology.org/volumes/P18-1/.pdf,None,ACL 2018
1,Probabilistic FastText for Multi-Sense Word Em...,"Ben Athiwaratkun, Andrew Gordon Wilson, Anima ...",https://aclanthology.org/P18-1001/.pdf,None,ACL 2018
2,A La Carte Embedding: Cheap but Effective Indu...,"Mikhail Khodak, Nikunj Saunshi, Yingyu Liang, ...",https://aclanthology.org/P18-1002/.pdf,None,ACL 2018
3,Unsupervised Learning of Distributional Relati...,"Shoaib Jameel, Zied Bouraoui, Steven Schockaert",https://aclanthology.org/P18-1003/.pdf,None,ACL 2018
4,Explicit Retrofitting of Distributional Word V...,"Goran Glavas, Ivan Vulic",https://aclanthology.org/P18-1004/.pdf,None,ACL 2018


In [146]:
df_papers = df_papers.drop(index=[0])

In [147]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Probabilistic FastText for Multi-Sense Word Em...,"Ben Athiwaratkun, Andrew Gordon Wilson, Anima ...",https://aclanthology.org/P18-1001/.pdf,None,ACL 2018
2,A La Carte Embedding: Cheap but Effective Indu...,"Mikhail Khodak, Nikunj Saunshi, Yingyu Liang, ...",https://aclanthology.org/P18-1002/.pdf,None,ACL 2018
3,Unsupervised Learning of Distributional Relati...,"Shoaib Jameel, Zied Bouraoui, Steven Schockaert",https://aclanthology.org/P18-1003/.pdf,None,ACL 2018
4,Explicit Retrofitting of Distributional Word V...,"Goran Glavas, Ivan Vulic",https://aclanthology.org/P18-1004/.pdf,None,ACL 2018
5,Unsupervised Neural Machine Translation with W...,"Zhen Yang, Wei Chen, Feng Wang, Bo Xu",https://aclanthology.org/P18-1005/.pdf,None,ACL 2018


In [148]:
save_to_database(df_papers, conference_name, DB_PATH)

256개의 논문이 ACL 2018에 저장되었습니다.


# 2017

In [150]:
url = 'https://dblp.org/db/conf/acl/acl2017-1.html'
DB_PATH = "con_db/ACL_conference_2017.db"
conference_name = 'ACL 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [151]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [152]:
df_papers = get_www_papers('html/ACL_2017_accepted_papers.html', conference_name)

In [153]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 55th Annual Meeting of the ...,"Regina Barzilay, Min-Yen Kan",https://aclanthology.org/volumes/P17-1/.pdf,None,ACL 2017
1,Adversarial Multi-task Learning for Text Class...,"Pengfei Liu, Xipeng Qiu, Xuanjing Huang",https://aclanthology.org/P17-1001.pdf,None,ACL 2017
2,Neural End-to-End Learning for Computational A...,"Steffen Eger, Johannes Daxenberger, Iryna Gure...",https://aclanthology.org/P17-1002.pdf,None,ACL 2017
3,Neural Symbolic Machines: Learning Semantic Pa...,"Chen Liang, Jonathan Berant, Quoc V. Le, Kenne...",https://aclanthology.org/P17-1003.pdf,None,ACL 2017
4,Neural Relation Extraction with Multi-lingual ...,"Yankai Lin, Zhiyuan Liu, Maosong Sun",https://aclanthology.org/P17-1004.pdf,None,ACL 2017


In [154]:
df_papers = df_papers.drop(index=[0])

In [155]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Adversarial Multi-task Learning for Text Class...,"Pengfei Liu, Xipeng Qiu, Xuanjing Huang",https://aclanthology.org/P17-1001.pdf,None,ACL 2017
2,Neural End-to-End Learning for Computational A...,"Steffen Eger, Johannes Daxenberger, Iryna Gure...",https://aclanthology.org/P17-1002.pdf,None,ACL 2017
3,Neural Symbolic Machines: Learning Semantic Pa...,"Chen Liang, Jonathan Berant, Quoc V. Le, Kenne...",https://aclanthology.org/P17-1003.pdf,None,ACL 2017
4,Neural Relation Extraction with Multi-lingual ...,"Yankai Lin, Zhiyuan Liu, Maosong Sun",https://aclanthology.org/P17-1004.pdf,None,ACL 2017
5,Learning Structured Natural Language Represent...,"Jianpeng Cheng, Siva Reddy, Vijay A. Saraswat,...",https://aclanthology.org/P17-1005.pdf,None,ACL 2017


In [156]:
save_to_database(df_papers, conference_name, DB_PATH)

195개의 논문이 ACL 2017에 저장되었습니다.


# 2016

In [170]:
from bs4 import BeautifulSoup
import pandas as pd

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 정보가 포함된 li 태그 찾기
    for paper in soup.find_all('li', class_='entry'):
        # 제목 찾기
        title_tag = paper.find('span', class_='title')
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        author_tags = paper.find_all('span', itemprop='author')
        authors_cleaned = ", ".join([author.text.strip() for author in author_tags]) if author_tags else "Unknown"

        # PDF 링크 만들기
        pdf_link = None
        
        # ACL Anthology 링크 찾기
        aclanthology_link = paper.find('a', href=True)
        if aclanthology_link and "aclanthology.org" in aclanthology_link['href']:
            pdf_link = aclanthology_link['href'] + ".pdf"
        
        # DOI 링크 찾기
        elif aclanthology_link and "doi.org" in aclanthology_link['href']:
            # DOI 링크에서 논문 코드만 추출
            doi_link = aclanthology_link['href']
            paper_code = doi_link.split('/')[-1]  # DOI에서 논문 코드만 추출
            paper_code = paper_code[0].upper() + paper_code[1:]  # 첫 글자를 대문자로 변환
            pdf_link = f"https://aclanthology.org/{paper_code}.pdf"
        
        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            "code_url":None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [157]:
url = 'https://dblp.org/db/conf/acl/acl2016-1.html'
DB_PATH = "con_db/ACL_conference_2016.db"
conference_name = 'ACL 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [158]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [172]:
df_papers = get_www_papers('html/ACL_2016_accepted_papers.html', conference_name)

In [173]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 54th Annual Meeting of the ...,Unknown,https://aclanthology.org/volumes/P16-1/.pdf,None,ACL 2016
1,Noise reduction and targeted exploration in im...,"James Goodman, Andreas Vlachos, Jason Naradowsky",https://aclanthology.org/P16-1001.pdf,None,ACL 2016
2,Data Recombination for Neural Semantic Parsing.,"Robin Jia, Percy Liang",https://aclanthology.org/P16-1002.pdf,None,ACL 2016
3,Inferring Logical Forms From Denotations.,"Panupong Pasupat, Percy Liang",https://aclanthology.org/P16-1003.pdf,None,ACL 2016
4,Language to Logical Form with Neural Attention.,"Li Dong, Mirella Lapata",https://aclanthology.org/P16-1004.pdf,None,ACL 2016


In [174]:
df_papers = df_papers.drop(index=[0])

In [175]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Noise reduction and targeted exploration in im...,"James Goodman, Andreas Vlachos, Jason Naradowsky",https://aclanthology.org/P16-1001.pdf,None,ACL 2016
2,Data Recombination for Neural Semantic Parsing.,"Robin Jia, Percy Liang",https://aclanthology.org/P16-1002.pdf,None,ACL 2016
3,Inferring Logical Forms From Denotations.,"Panupong Pasupat, Percy Liang",https://aclanthology.org/P16-1003.pdf,None,ACL 2016
4,Language to Logical Form with Neural Attention.,"Li Dong, Mirella Lapata",https://aclanthology.org/P16-1004.pdf,None,ACL 2016
5,Unsupervised Person Slot Filling based on Grap...,"Dian Yu, Heng Ji",https://aclanthology.org/P16-1005.pdf,None,ACL 2016


In [176]:
save_to_database(df_papers, conference_name, DB_PATH)

231개의 논문이 ACL 2016에 저장되었습니다.


# 2015

In [177]:
url = 'https://dblp.org/db/conf/acl/acl2015-1.html'
DB_PATH = "con_db/ACL_conference_2015.db"
conference_name = 'ACL 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [178]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [179]:
df_papers = get_www_papers('html/ACL_2015_accepted_papers.html', conference_name)

In [180]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 53rd Annual Meeting of the ...,Unknown,https://aclanthology.org/volumes/P15-1/.pdf,None,ACL 2015
1,On Using Very Large Target Vocabulary for Neur...,"Sébastien Jean, KyungHyun Cho, Roland Memisevi...",https://aclanthology.org/P15-1001.pdf,None,ACL 2015
2,Addressing the Rare Word Problem in Neural Mac...,"Thang Luong, Ilya Sutskever, Quoc V. Le, Oriol...",https://aclanthology.org/P15-1002.pdf,None,ACL 2015
3,Encoding Source Language with Convolutional Ne...,"Fandong Meng, Zhengdong Lu, Mingxuan Wang, Han...",https://aclanthology.org/P15-1003.pdf,None,ACL 2015
4,Statistical Machine Translation Features with ...,"Hendra Setiawan, Zhongqiang Huang, Jacob Devli...",https://aclanthology.org/P15-1004.pdf,None,ACL 2015


In [181]:
df_papers = df_papers.drop(index=[0])

In [182]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,On Using Very Large Target Vocabulary for Neur...,"Sébastien Jean, KyungHyun Cho, Roland Memisevi...",https://aclanthology.org/P15-1001.pdf,None,ACL 2015
2,Addressing the Rare Word Problem in Neural Mac...,"Thang Luong, Ilya Sutskever, Quoc V. Le, Oriol...",https://aclanthology.org/P15-1002.pdf,None,ACL 2015
3,Encoding Source Language with Convolutional Ne...,"Fandong Meng, Zhengdong Lu, Mingxuan Wang, Han...",https://aclanthology.org/P15-1003.pdf,None,ACL 2015
4,Statistical Machine Translation Features with ...,"Hendra Setiawan, Zhongqiang Huang, Jacob Devli...",https://aclanthology.org/P15-1004.pdf,None,ACL 2015
5,Describing Images using Inferred Visual Depend...,"Desmond Elliott, Arjen P. de Vries",https://aclanthology.org/P15-1005.pdf,None,ACL 2015


In [183]:
save_to_database(df_papers, conference_name, DB_PATH)

174개의 논문이 ACL 2015에 저장되었습니다.


# 2014

In [184]:
url = 'https://dblp.org/db/conf/acl/acl2014-1.html'
DB_PATH = "con_db/ACL_conference_2014.db"
conference_name = 'ACL 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [185]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ACL_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [186]:
df_papers = get_www_papers('html/ACL_2014_accepted_papers.html', conference_name)

In [187]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 52nd Annual Meeting of the ...,Unknown,https://aclanthology.org/volumes/P14-1/.pdf,None,ACL 2014
1,Learning Ensembles of Structured Prediction Ru...,"Corinna Cortes, Vitaly Kuznetsov, Mehryar Mohri",https://aclanthology.org/P14-1001.pdf,None,ACL 2014
2,Representation Learning for Text-level Discour...,"Yangfeng Ji, Jacob Eisenstein",https://aclanthology.org/P14-1002.pdf,None,ACL 2014
3,Text-level Discourse Dependency Parsing.,"Sujian Li, Liang Wang, Ziqiang Cao, Wenjie Li",https://aclanthology.org/P14-1003.pdf,None,ACL 2014
4,Discovering Latent Structure in Task-Oriented ...,"Ke Zhai, Jason D. Williams",https://aclanthology.org/P14-1004.pdf,None,ACL 2014


In [188]:
df_papers = df_papers.drop(index=[0])

In [189]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Learning Ensembles of Structured Prediction Ru...,"Corinna Cortes, Vitaly Kuznetsov, Mehryar Mohri",https://aclanthology.org/P14-1001.pdf,None,ACL 2014
2,Representation Learning for Text-level Discour...,"Yangfeng Ji, Jacob Eisenstein",https://aclanthology.org/P14-1002.pdf,None,ACL 2014
3,Text-level Discourse Dependency Parsing.,"Sujian Li, Liang Wang, Ziqiang Cao, Wenjie Li",https://aclanthology.org/P14-1003.pdf,None,ACL 2014
4,Discovering Latent Structure in Task-Oriented ...,"Ke Zhai, Jason D. Williams",https://aclanthology.org/P14-1004.pdf,None,ACL 2014
5,Learning Structured Perceptrons for Coreferenc...,"Anders Björkelund, Jonas Kuhn",https://aclanthology.org/P14-1005.pdf,None,ACL 2014


In [190]:
save_to_database(df_papers, conference_name, DB_PATH)

147개의 논문이 ACL 2014에 저장되었습니다.


In [ ]:
s